# 04_Validation

Run on Databricks serverless with the uploaded Olist files. Original team work: review, execute, validate and document results.

In [ ]:
CATALOG = "workspace"
SCHEMA = "ecommerce_bigdata"
VOLUME = "raw_data"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print('Raw data path:', RAW_PATH)

In [ ]:
from pyspark.sql import functions as F
p=f"{CATALOG}.{SCHEMA}"
checks=[]
for t in ['bronze_orders','silver_orders','silver_order_items','gold_order_items_fact','gold_monthly_sales','gold_category_sales','gold_state_sales','gold_delivery_summary','gold_payment_summary']:
    n=spark.table(f'{p}.{t}').count(); checks.append((t,n)); print(t,n)
assert all(n>0 for _,n in checks), 'A required table is empty'
assert spark.table(f'{p}.silver_orders').groupBy('order_id').count().filter('count > 1').count()==0
assert spark.table(f'{p}.silver_order_items').filter('price < 0 OR freight_value < 0').count()==0
assert spark.table(f'{p}.gold_order_items_fact').filter('order_id IS NULL').count()==0
print('Core validation checks passed. Capture screenshot of this output.')

In [ ]:
# Optional: export Gold tables as CSV for Power BI Desktop via Databricks UI.
# In Catalog, open each gold_* table and use Download (if available).
# For larger tables use a controlled cloud export instead.